<a href="https://colab.research.google.com/github/binh-star/app_test_stress_LoveC/blob/main/app_test_stress_LoveC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded=files.upload()

Saving StressLevelDataset.csv to StressLevelDataset.csv


In [2]:
import pandas as pd
data=pd.read_csv('StressLevelDataset.csv')

In [6]:
# =========================================================
# AI STRESS ANALYSIS SYSTEM
# FIXED LOGIC VERSION
# PERCEPTRON + GRADIO + HEALING UI
# =========================================================

# CÀI:
# pip install pandas numpy scikit-learn gradio

import pandas as pd
import numpy as np
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# =========================================================
# 1. LOAD DATASET
# =========================================================

df = pd.read_csv("StressLevelDataset.csv")

# =========================================================
# 2. FEATURES
# =========================================================

features = [
    "anxiety_level",
    "self_esteem",
    "mental_health_history",
    "depression",
    "headache",
    "blood_pressure",
    "sleep_quality",
    "breathing_problem",
    "noise_level",
    "living_conditions",
    "safety",
    "basic_needs",
    "academic_performance",
    "study_load",
    "teacher_student_relationship",
    "future_career_concerns",
    "social_support",
    "peer_pressure",
    "extracurricular_activities",
    "bullying"
]

X = df[features]
y = df["stress_level"]

# =========================================================
# 3. TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =========================================================
# 4. SCALE DATA
# =========================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================================================
# 5. OPTIMIZE PERCEPTRON
# =========================================================

param_grid = {
    "max_iter": [2000, 3000, 5000],
    "eta0": [0.0001, 0.001, 0.01],
    "penalty": [None, "l2", "l1"]
}

base_model = Perceptron(random_state=42)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

model = grid_search.best_estimator_

model.fit(X_train, y_train)

# =========================================================
# 6. ACCURACY
# =========================================================

pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)

print("Độ chính xác:", accuracy)

# =========================================================
# 7. MIN MAX FROM DATASET
# =========================================================

feature_ranges = {}

for feature in features:

    feature_ranges[feature] = {
        "min": int(df[feature].min()),
        "max": int(df[feature].max())
    }

# =========================================================
# 8. OVERALL STRESS
# =========================================================

def overall_stress(prediction):

    if prediction == 0:
        return "Stress thấp"

    elif prediction == 1:
        return "Stress trung bình"

    else:
        return "Stress cao"

# =========================================================
# 9. ANALYZE STRESS TYPES
# =========================================================

def analyze_types(data):

    results = []
    advice = []

    # =====================================================
    # MAX VALUES
    # =====================================================

    max_self_esteem = feature_ranges["self_esteem"]["max"]
    max_academic = feature_ranges["academic_performance"]["max"]
    max_social_support = feature_ranges["social_support"]["max"]
    max_living = feature_ranges["living_conditions"]["max"]
    max_safety = feature_ranges["safety"]["max"]
    max_sleep = feature_ranges["sleep_quality"]["max"]

    # =====================================================
    # SCORES
    # =====================================================

    academic_score = (
        data["study_load"] +
        data["future_career_concerns"] +
        (max_academic - data["academic_performance"])
    ) / 3

    social_score = (
        data["peer_pressure"] +
        data["bullying"] +
        (max_social_support - data["social_support"])
    ) / 3

    psychological_score = (
        data["anxiety_level"] +
        data["depression"] +
        (max_self_esteem - data["self_esteem"])
    ) / 3

    environmental_score = (
        data["noise_level"] +
        (max_living - data["living_conditions"]) +
        (max_safety - data["safety"])
    ) / 3

    physical_score = (
        data["headache"] +
        data["blood_pressure"] +
        data["breathing_problem"] +
        (max_sleep - data["sleep_quality"])
    ) / 4

    # =====================================================
    # LEVEL
    # =====================================================

    def level(score):

        if score < 3:
            return "Thấp"

        elif score < 6:
            return "Trung bình"

        else:
            return "Cao"

    # =====================================================
    # RESULTS
    # =====================================================

    results.append(
        f"📚 Stress học tập: {level(academic_score)} | Điểm: {academic_score:.2f}"
    )

    results.append(
        f"👥 Stress xã hội: {level(social_score)} | Điểm: {social_score:.2f}"
    )

    results.append(
        f"🧠 Stress tâm lý: {level(psychological_score)} | Điểm: {psychological_score:.2f}"
    )

    results.append(
        f"🌍 Stress môi trường: {level(environmental_score)} | Điểm: {environmental_score:.2f}"
    )

    results.append(
        f"💪 Stress thể chất: {level(physical_score)} | Điểm: {physical_score:.2f}"
    )

    # =====================================================
    # PERSONALIZED ADVICE
    # =====================================================

    if academic_score >= 6:

        advice.append("""
📚 Lời khuyên cho stress học tập:

Bạn đang chịu khá nhiều áp lực từ học tập hoặc tương lai nghề nghiệp. Hãy thử chia nhỏ mục tiêu học tập và nghỉ ngơi hợp lý thay vì học liên tục trong thời gian dài.

Việc quản lý thời gian tốt hơn và chia sẻ khó khăn với bạn bè hoặc giảng viên sẽ giúp bạn giảm áp lực đáng kể.
""")

    if social_score >= 6:

        advice.append("""
👥 Lời khuyên cho stress xã hội:

Bạn có thể đang chịu ảnh hưởng từ áp lực bạn bè hoặc thiếu sự hỗ trợ xã hội. Hãy ưu tiên các mối quan hệ tích cực và tránh môi trường khiến bạn cảm thấy tiêu cực.

Chia sẻ cảm xúc với người đáng tin cậy sẽ giúp tinh thần của bạn cân bằng hơn.
""")

    if psychological_score >= 6:

        advice.append("""
🧠 Lời khuyên cho stress tâm lý:

Lo âu hoặc cảm xúc tiêu cực đang ảnh hưởng khá nhiều đến tinh thần của bạn. Hãy thử thư giãn bằng thiền, hít thở sâu hoặc vận động nhẹ mỗi ngày.

Nếu cảm xúc tiêu cực kéo dài, bạn nên tìm sự hỗ trợ từ chuyên gia tâm lý.
""")

    if environmental_score >= 6:

        advice.append("""
🌍 Lời khuyên cho stress môi trường:

Môi trường sống hoặc học tập hiện tại có thể đang khiến bạn cảm thấy mệt mỏi. Hãy thử tạo không gian yên tĩnh, sạch sẽ và thoải mái hơn.

Một môi trường tích cực sẽ giúp bạn cải thiện cảm xúc và khả năng tập trung.
""")

    if physical_score >= 6:

        advice.append("""
💪 Lời khuyên cho stress thể chất:

Cơ thể bạn có thể đang chịu ảnh hưởng bởi stress thông qua các triệu chứng như đau đầu hoặc mất ngủ. Hãy ưu tiên ngủ đủ giấc và duy trì vận động nhẹ thường xuyên.

Nếu các triệu chứng kéo dài, hãy tìm sự hỗ trợ từ bác sĩ hoặc chuyên gia y tế.
""")

    # =====================================================
    # FINAL OUTPUT
    # =====================================================

    final_text = "\n".join(results)

    if len(advice) > 0:

        final_text += "\n\n# 🌿 LỜI KHUYÊN CÁ NHÂN HÓA\n\n"

        final_text += "\n\n".join(advice)

    return final_text

# =========================================================
# 10. MAIN PREDICT FUNCTION
# =========================================================

def predict_stress(
    anxiety_level,
    self_esteem,
    mental_health_history,
    depression,
    headache,
    blood_pressure,
    sleep_quality,
    breathing_problem,
    noise_level,
    living_conditions,
    safety,
    basic_needs,
    academic_performance,
    study_load,
    teacher_student_relationship,
    future_career_concerns,
    social_support,
    peer_pressure,
    extracurricular_activities,
    bullying
):

    input_data = {
        "anxiety_level": anxiety_level,
        "self_esteem": self_esteem,
        "mental_health_history": mental_health_history,
        "depression": depression,
        "headache": headache,
        "blood_pressure": blood_pressure,
        "sleep_quality": sleep_quality,
        "breathing_problem": breathing_problem,
        "noise_level": noise_level,
        "living_conditions": living_conditions,
        "safety": safety,
        "basic_needs": basic_needs,
        "academic_performance": academic_performance,
        "study_load": study_load,
        "teacher_student_relationship": teacher_student_relationship,
        "future_career_concerns": future_career_concerns,
        "social_support": social_support,
        "peer_pressure": peer_pressure,
        "extracurricular_activities": extracurricular_activities,
        "bullying": bullying
    }

    input_df = pd.DataFrame([input_data])

    scaled = scaler.transform(input_df)

    prediction = model.predict(scaled)[0]

    overall = overall_stress(prediction)

    # =====================================================
    # MAX VALUES
    # =====================================================

    max_self_esteem = feature_ranges["self_esteem"]["max"]
    max_social_support = feature_ranges["social_support"]["max"]
    max_academic = feature_ranges["academic_performance"]["max"]
    max_sleep = feature_ranges["sleep_quality"]["max"]
    max_living = feature_ranges["living_conditions"]["max"]
    max_safety = feature_ranges["safety"]["max"]

    # =====================================================
    # CHECK GOOD CONDITIONS
    # =====================================================

    positive_ok = (
        self_esteem >= max_self_esteem * 0.7 and
        social_support >= max_social_support * 0.7 and
        academic_performance >= max_academic * 0.7 and
        sleep_quality >= max_sleep * 0.7 and
        living_conditions >= max_living * 0.7 and
        safety >= max_safety * 0.7
    )

    negative_low = (
        anxiety_level <= 2 and
        depression <= 2 and
        headache <= 2 and
        blood_pressure <= 2 and
        breathing_problem <= 2 and
        noise_level <= 2 and
        peer_pressure <= 2 and
        bullying <= 2 and
        study_load <= 2 and
        future_career_concerns <= 2
    )

    # =====================================================
    # NO STRESS
    # =====================================================

    if positive_ok and negative_low:

        final_result = """
# 🌿 KẾT QUẢ PHÂN TÍCH

## 🎉 Bạn ổn và chúc mừng bạn!

Hiện tại hệ thống không phát hiện dấu hiệu stress đáng lo ngại.

Các chỉ số cảm xúc, môi trường sống và sức khỏe tinh thần của bạn đang ở mức ổn định.

Hãy tiếp tục duy trì thói quen sinh hoạt lành mạnh, nghỉ ngơi hợp lý và giữ cân bằng giữa học tập, công việc và cuộc sống 💙
"""

    # =====================================================
    # HAVE STRESS
    # =====================================================

    else:

        type_analysis = analyze_types(input_data)

        final_result = f"""
# 🧠 KẾT QUẢ PHÂN TÍCH STRESS

📊 Mức độ stress tổng thể: {overall}

# 📌 PHÂN LOẠI CÁC LOẠI STRESS

{type_analysis}
"""

    return final_result

# =========================================================
# 11. HEALING UI
# =========================================================

custom_css = """

body {
    background: linear-gradient(
        135deg,
        #dbeafe,
        #dcfce7,
        #f5d0fe
    );
}

.gradio-container {
    font-family: 'Segoe UI', sans-serif;
}

h1, h2, h3 {
    color: #1e3a8a;
}

.gr-button {
    background: linear-gradient(
        90deg,
        #60a5fa,
        #86efac
    ) !important;

    color: white !important;

    border-radius: 16px !important;

    border: none !important;

    font-size: 18px !important;
}

"""

# =========================================================
# 12. GRADIO UI
# =========================================================

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=custom_css,
    title="AI Stress Analysis"
) as app:

    gr.Markdown("""
# 🧠 AI STRESS ANALYSIS SYSTEM

🌿 Hệ thống AI hỗ trợ phân tích stress và sức khỏe tinh thần.
""")

    inputs = []

    for feature in features:

        if feature == "mental_health_history":

            comp = gr.Radio(
                choices=[("Không",0), ("Có",1)],
                label="Tiền sử vấn đề tâm lý"
            )

        else:

            comp = gr.Slider(
                minimum=feature_ranges[feature]["min"],
                maximum=feature_ranges[feature]["max"],
                step=1,
                label=feature.replace("_", " ").title()
            )

        inputs.append(comp)

    output = gr.Markdown()

    btn = gr.Button("🔍 Phân tích stress")

    btn.click(
        fn=predict_stress,
        inputs=inputs,
        outputs=output
    )

# =========================================================
# 13. RUN APP
# =========================================================

app.launch(share=True)

Độ chính xác: 0.8772727272727273


/tmp/ipykernel_2658/3069965370.py:480: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_2658/3069965370.py:480: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2230ffd011932e4c8a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
